# R2000 CR-HSDPA (r=0.50) - R2000 200-Epoch Single Run
**Thesis Topic:** A Modified Attention-PestNet with Channel-Reduced Hierarchical Scaled Dot-Product Attention for Efficient Insect Pest Detection  
**Authors:** Lean Adrian Murillo, James Oliver C. Mendoza, DM Rashid P. Ferrer, Charisse P. Barbosa (University of Mindanao)  
**Target Architecture:** Channel-Reduced HSDPA (`cfg/cr_hsdpa_050.yaml`, r=0.50)  
**Dataset:** R2000-PestNet (`dmrashidpferrer/r2000-pestnet`, 16 rice-pest classes)  
**Training:** 200 epochs single run from scratch (SGD `lr0=0.01`, `momentum=0.9`, `batch=32`, `imgsz=640`, Dual T4 — fits 12h window)  
**Branch:** `HSDPA-DM-Testing` (reuses `90377Sednaaa/HSDPA`, account-agnostic via Kaggle CLI).  
---
Run cells top-to-bottom exactly once (single `model.train()` cell). Classes auto-discovered from dataset at runtime into `cfg/r2000.yaml`.


In [ ]:
# 1. Setup repository and install ultralytics in editable mode
# Works on any Kaggle account (authenticated via Kaggle CLI).
import os
os.environ["WANDB_MODE"] = "disabled"
import sys

REPO_URL = "https://github.com/90377Sednaaa/HSDPA.git"
BRANCH = "HSDPA-DM-Testing"
REPO_DIR = "/kaggle/working/HSDPA"
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q -e .

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Environment ready at:", REPO_DIR, "| branch:", BRANCH)


In [ ]:
# 2. Check GPU environment and CUDA capabilities
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
device_count = torch.cuda.device_count()
print("GPU Count:", device_count)
for i in range(device_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# 3. Verify R2000 dataset + auto-discover classes + write cfg/r2000.yaml
# Attach in Kaggle UI or CLI: dmrashidpferrer/r2000-pestnet
import os, glob, yaml

DATA_ROOT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "images" in dirs and os.path.exists(os.path.join(root, "images/train")):
        DATA_ROOT = root
        break

if not DATA_ROOT:
    raise FileNotFoundError("Could not locate R2000 dataset in /kaggle/input. Attach dmrashidpferrer/r2000-pestnet.")

print(f"Dataset root verified at: {DATA_ROOT}")
train_count = len(os.listdir(os.path.join(DATA_ROOT, "images/train")))
val_dir = os.path.join(DATA_ROOT, "images/val")
val_count = len(os.listdir(val_dir)) if os.path.exists(val_dir) else 0
test_dir = os.path.join(DATA_ROOT, "images/test")
test_count = len(os.listdir(test_dir)) if os.path.exists(test_dir) else 0
print(f"Train images: {train_count} (Expected ~1225)")
print(f"Validation images: {val_count} (Expected ~298)")
print(f"Test images: {test_count} (Expected ~523)")

# Auto-discover class names: prefer a data yaml shipped with the dataset
nc, names = 16, {i: f"class_{i}" for i in range(16)}
shipped = glob.glob(os.path.join(DATA_ROOT, "*.yaml")) + glob.glob(os.path.join(DATA_ROOT, "*.yml")) \
    + glob.glob(os.path.join(DATA_ROOT, "**", "data.yaml"), recursive=True)
for yp in shipped:
    try:
        with open(yp) as f:
            d = yaml.safe_load(f)
        if isinstance(d, dict) and ("names" in d or "nc" in d):
            raw = d.get("names", [])
            if isinstance(raw, dict):
                names = {int(k): str(v) for k, v in raw.items()}
            elif isinstance(raw, list):
                names = {i: str(v) for i, v in enumerate(raw)}
            nc = int(d.get("nc", len(names)))
            print(f"Discovered class config from {yp}: nc={nc}")
            break
    except Exception as e:
        print(f"Skipping {yp}: {e}")

print(f"Using nc={nc}, classes: {list(names.values())[:5]}...")
r2000_cfg = {"path": DATA_ROOT, "train": "images/train", "val": "images/val", "test": "images/test",
            "nc": nc, "names": names}
with open("cfg/r2000.yaml", "w") as f:
    yaml.dump(r2000_cfg, f, default_flow_style=False, sort_keys=False)
print("cfg/r2000.yaml written and ready!")


In [ ]:
# 4. Train CR-HSDPA (r=0.50) on R2000 (200 epochs, single run)
import torch
import os, glob, time, yaml
os.environ["WANDB_MODE"] = "disabled"
from ultralytics import YOLO

MODEL_CFG_SRC = "cfg/cr_hsdpa_050.yaml"
PROJECT_NAME = "dataset_r2000_r050"
EXP_NAME = "r2000_cr050_200ep"

# Patch model yaml nc to discovered R2000 nc (source cfgs are nc=102 for IP102)
with open("cfg/r2000.yaml") as f:
    r2000_nc = int(yaml.safe_load(f)["nc"])
with open(MODEL_CFG_SRC) as f:
    mcfg = yaml.safe_load(f)
mcfg["nc"] = r2000_nc
MODEL_CFG = f"/tmp/{EXP_NAME}_model.yaml"
with open(MODEL_CFG, "w") as f:
    yaml.dump(mcfg, f, default_flow_style=False, sort_keys=False)
print(f"Model cfg patched for R2000: {MODEL_CFG_SRC} -> {MODEL_CFG} (nc={r2000_nc})")

# Resume only if THIS run was interrupted (same project/name -> last.pt)
resume_ckpts = glob.glob(f"{PROJECT_NAME}/{EXP_NAME}/weights/last.pt")
if resume_ckpts:
    print(f"Resuming interrupted run from: {resume_ckpts[0]}")
    model = YOLO(resume_ckpts[0])
    resume_flag = True
else:
    print(f"Initializing fresh CR-HSDPA (r=0.50) model from {MODEL_CFG}")
    model = YOLO(MODEL_CFG)
    resume_flag = False

# Multi-GPU support for dual T4
device_arg = [0, 1] if torch.cuda.device_count() >= 2 else 0

train_start_wall = time.time()

# Single 200-epoch run (fits 12h T4 window on R2000 ~2k images)
results = model.train(
    data="cfg/r2000.yaml",
    epochs=200,
    batch=32,
    imgsz=640,
    device=device_arg,
    optimizer="SGD",
    lr0=0.01,
    momentum=0.9,
    amp=True,
    save_period=25,
    project=PROJECT_NAME,
    name=EXP_NAME,
    exist_ok=True,
    workers=4,
    resume=resume_flag,
    verbose=True
)

train_elapsed_sec = time.time() - train_start_wall
train_elapsed_hrs = train_elapsed_sec / 3600.0
print(f"\nCR-HSDPA (r=0.50) R2000 200-epoch training completed in {train_elapsed_hrs:.2f} hours ({train_elapsed_sec:.1f} s).")
os.makedirs(f"{PROJECT_NAME}/{EXP_NAME}", exist_ok=True)
with open(f"{PROJECT_NAME}/{EXP_NAME}/training_time.txt", "w") as f:
    f.write(f"{train_elapsed_sec:.2f}\n")


In [ ]:
# 5. Comprehensive Thesis Evaluation & Efficiency Benchmark (Section 2.7 Protocol)
import time, os, glob
import torch
import pandas as pd
from ultralytics import YOLO

best_ckpt = f"{PROJECT_NAME}/{EXP_NAME}/weights/best.pt"
if os.path.exists(best_ckpt):
    print(f"=== Evaluating CR-HSDPA (r=0.50) R2000 Best Checkpoint (200 Epochs): {best_ckpt} ===\n")
    eval_model = YOLO(best_ckpt)

    # 1. Detection Performance (mAP@50, mAP@50:95, Precision, Recall)
    val_results = eval_model.val(data="cfg/r2000.yaml", split="val", batch=32, imgsz=640, device=0)
    p = val_results.box.mp
    r = val_results.box.mr
    map50 = val_results.box.map50
    map50_95 = val_results.box.map

    # 2. Computational Efficiency Benchmark (Section 2.7: FP32, batch 1, 20 warmup, 200 timed runs)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    bench_model = eval_model.model.to(device).float()
    bench_model.eval()
    dummy_input = torch.randn(1, 3, 640, 640, device=device)

    with torch.no_grad():
        for _ in range(20):
            _ = bench_model(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()
        for _ in range(200):
            _ = bench_model(dummy_input)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        total_time = time.perf_counter() - start_time

    latency_ms = (total_time / 200.0) * 1000.0
    fps = 200.0 / total_time
    total_params = sum(x.numel() for x in eval_model.model.parameters())
    model_size_mb = os.path.getsize(best_ckpt) / (1024 * 1024)

    time_file = f"{PROJECT_NAME}/{EXP_NAME}/training_time.txt"
    train_time_hrs = None
    if os.path.exists(time_file):
        with open(time_file, "r") as f:
            train_time_hrs = float(f.read().strip()) / 3600.0

    print("\n" + "=" * 65)
    print("   THESIS EVALUATION METRICS SUMMARY - CR-HSDPA (r=0.50) R2000 200EP (Section 2.7)")
    print("=" * 65)
    print(f"  Precision (P):          {p:.4f} ({p*100:.2f}%)")
    print(f"  Recall (R):             {r:.4f} ({r*100:.2f}%)")
    print(f"  mAP@50:                 {map50:.4f} ({map50*100:.2f}%)")
    print(f"  mAP@50:95:              {map50_95:.4f} ({map50_95*100:.2f}%)")
    print(f"  Parameters:             {total_params:,}")
    if train_time_hrs:
        print(f"  Training Time:          {train_time_hrs:.2f} hours (200 epochs)")
    print(f"  Inference Latency:      {latency_ms:.2f} ms")
    print(f"  Throughput (FPS):       {fps:.2f} FPS")
    print(f"  Model File Size:        {model_size_mb:.2f} MB")
    print("=" * 65)

    summary_df = pd.DataFrame([{
        "Model": "CR-HSDPA (r=0.50) R2000 (200ep)",
        "Precision": p,
        "Recall": r,
        "mAP@50": map50,
        "mAP@50-95": map50_95,
        "Parameters": total_params,
        "Training_Time_Hours": round(train_time_hrs, 2) if train_time_hrs else "N/A",
        "Latency_ms": latency_ms,
        "FPS": fps,
        "ModelSize_MB": model_size_mb
    }])
    out_csv = f"{PROJECT_NAME}/{EXP_NAME}/thesis_metrics_summary_r2000_cr050_200ep.csv"
    summary_df.to_csv(out_csv, index=False)
    print(f"Saved R2000 metrics summary to: {out_csv}")
else:
    print(f"Checkpoint {best_ckpt} not found.")
